In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "6"

In [2]:
import time
from functools import partial
from datetime import datetime, timedelta

import datasets as hfds
import torch
import pandas as pd
import torchvision.transforms.v2 as v2
from torch.utils import flop_counter
from matplotlib import pyplot as plt
from tqdm import tqdm

import flat_mae.transforms as flat_transforms
import flat_mae.data as flat_data
import flat_mae.masking as flat_masking
import flat_mae.models_mae as models_mae

In [3]:
torch.backends.cudnn.benchmark = True

In [4]:
plt.style.use("../clane.v2.mplstyle")
PLOTW = 3.25  # 6.75 two column width, 0.25 pad

In [5]:
def prefix_crop(sample, num_frames: int = 16):
    sample["bold"] = sample["bold"][:num_frames]
    return sample


def expand_mask(sample):
    sample["mask"] = sample["mask"][None, None]
    return sample


def make_transform(space: str = "flat"):
    # hack, have to redefine to exclude center crop bc mni clips is only 16 frames
    # (idk why ig I forgot to make it 24 frames like the others)
    transform = v2.Compose(
        [
            flat_transforms.ToTensor(),
            prefix_crop,
            flat_transforms.Normalize(mode="frame"),
            flat_transforms.Clip(vmax=3.0),
            flat_transforms.get_unmask(space),
            expand_mask,
        ]
    )
    return transform

In [6]:
input_spaces = ["schaefer400", "flat", "mni_cortex"]

In [7]:
dataset_root = "s3://medarc/fmri-datasets/eval"

datasets = {}
transforms = {}

for space in input_spaces:
    transforms[space] = transform = make_transform(space)
    dataset = hfds.load_dataset(
        "arrow",
        data_files=f"{dataset_root}/hcpya-clips.{space}.arrow/test/*.arrow",
        split="train",
    )
    dataset = flat_data.HFDataset(dataset, transforms[space])
    datasets[space] = dataset

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/36 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/18 [00:00<?, ?it/s]

In [8]:
img_sizes = {
    "schaefer400": (400, 1),
    "flat": (224, 560),
    "mni_cortex": (465, 512),
}

patch_sizes = {
    "schaefer400": 1,
    "flat": 16,
    "mni_cortex": (1, 512),
}

batch_size = 32
num_workers = 8
mask_ratio = 0.9
t_patch_size = 4

loaders = {}
for space in input_spaces:
    mask_fn = flat_masking.create_masking(
        name="tube",
        mask_ratio=mask_ratio,
        img_size=img_sizes[space],
        patch_size=patch_sizes[space],
        t_patch_size=t_patch_size,
    )
    collate_fn = partial(flat_masking.mask_collate, mask_fn=mask_fn)
    loaders[space] = torch.utils.data.DataLoader(
        datasets[space],
        batch_size=batch_size,
        collate_fn=collate_fn,
        shuffle=False,
        num_workers=num_workers,
        drop_last=True,
    )

In [9]:
device = torch.device("cuda")

In [10]:
models = {}

space_lrs = {
    "schaefer400": "3e-4",
    "flat": "1e-3",
    "mni_cortex": "1e-3",
}

for space in input_spaces:
    ckpt_path = f"output/input_space_v3/{space}_lr{space_lrs[space]}_1/pretrain/checkpoint-last.pth"
    models[space] = models_mae.MaskedAutoencoderViT.from_checkpoint(ckpt_path).to(device)

In [11]:
num_params = {}
for space in input_spaces:
    num_params[space] = num = sum(p.numel() for p in models[space].encoder.parameters())
    print(f"{space} {num / 1e6:.1f} M")

schaefer400 85.4 M
flat 86.2 M
mni_cortex 87.0 M


In [12]:
flops = {}
for space in input_spaces:
    batch = next(iter(loaders[space]))
    bold = batch["bold"].to(device)
    mask = batch["mask"].to(device)
    visible_mask = batch["visible_mask"].to(device)

    model = models[space]
    counter = flop_counter.FlopCounterMode(display=False)
    with counter:
        loss = model(
            bold,
            img_mask=mask,
            visible_mask=visible_mask,
            mask_ratio=None,
            with_state=False,
        )
    flops[space] = counter.get_total_flops() / batch_size
    print(f"{space} {flops[space] / 1e9:.0f}G")

schaefer400 89G
flat 92G
mni_cortex 116G


In [13]:
records = []

for run in range(5):
    for space in input_spaces:
        loader = loaders[space]
        model = models[space]

        total_time = 0.0
        total_samples = 0
        for ii, batch in tqdm(enumerate(loader)):
            bold = batch["bold"].to(device)
            mask = batch["mask"].to(device)
            visible_mask = batch["visible_mask"].to(device)
            torch.cuda.synchronize()
            tic = time.perf_counter()
            model.zero_grad()
            with torch.autocast(device_type=device.type, dtype=torch.float16):
                loss = model(
                    bold,
                    img_mask=mask,
                    visible_mask=visible_mask,
                    mask_ratio=None,
                    with_state=False,
                )
            loss.backward()
            torch.cuda.synchronize()
            step_time = time.perf_counter() - tic
            total_time += step_time
            total_samples += len(bold)
            # restart after one batch per worker warmup
            if (ii + 1) == num_workers:
                total_time = 0.0
                total_samples = 0

        records.append({"space": space, "run": run, "tput": total_samples / total_time})

throughput = pd.DataFrame.from_records(records)
throughput = throughput.groupby("space").agg({"tput": "median"})["tput"].to_dict()
print(throughput)

63it [00:03, 15.76it/s]
63it [00:11,  5.69it/s]
63it [00:19,  3.23it/s]
63it [00:03, 16.33it/s]
63it [00:11,  5.29it/s]
63it [00:19,  3.26it/s]
63it [00:03, 16.46it/s]
63it [00:11,  5.40it/s]
63it [00:19,  3.17it/s]
63it [00:03, 16.73it/s]
63it [00:11,  5.50it/s]
63it [00:19,  3.32it/s]
63it [00:03, 16.27it/s]
63it [00:11,  5.47it/s]
63it [00:19,  3.15it/s]

{'flat': 584.8388755329408, 'mni_cortex': 472.41557608546566, 'schaefer400': 615.0899250713404}


In [14]:
records = []

for run in range(5):
    for space in input_spaces:
        loader = loaders[space]
        total_time = 0.0
        total_samples = 0
        tic = time.perf_counter()
        for ii, batch in tqdm(enumerate(loader)):
            bold = batch["bold"].to(device)
            mask = batch["mask"].to(device)
            visible_mask = batch["visible_mask"].to(device)
            torch.cuda.synchronize()
            total_samples += len(bold)
            # restart after one batch per worker warmup
            if (ii + 1) == num_workers:
                tic = time.perf_counter()
                total_samples = 0
        total_time = time.perf_counter() - tic

        records.append({"space": space, "run": run, "tput": total_samples / total_time})

load_throughput = pd.DataFrame.from_records(records)
load_throughput = load_throughput.groupby("space").agg({"tput": "median"})["tput"].to_dict()
print(load_throughput)

63it [00:00, 109.22it/s]
63it [00:08,  7.80it/s]
63it [00:15,  4.02it/s]
63it [00:00, 103.46it/s]
63it [00:08,  7.63it/s]
63it [00:15,  4.12it/s]
63it [00:00, 93.55it/s] 
63it [00:08,  7.38it/s]
63it [00:15,  4.00it/s]
63it [00:00, 103.35it/s]
63it [00:08,  7.77it/s]
63it [00:15,  4.07it/s]
63it [00:00, 102.65it/s]
63it [00:08,  7.56it/s]
63it [00:15,  4.04it/s]

{'flat': 274.4230584085335, 'mni_cortex': 147.64160208891636, 'schaefer400': 3753.524353276825}


In [15]:
sample_dims = {
    "schaefer400": 400,
    "flat": 77763,
    "mni_cortex": 132032,
}

# assume fixed disk read bandwidth 400 MB/s which is fairly standard
# aggregate bandwidth across 8x gpus 3.2 GB/s which is about what I get in practice
disk_bandwidth = 400e6

ideal_load_throughput = {}
for space in input_spaces:
    sample_size = 16 * sample_dims[space] * 2
    ideal_load_throughput[space] = load_tput = disk_bandwidth / sample_size
    print(f"{space} {load_tput:.0f} sample / s")

schaefer400 31250 sample / s
flat 161 sample / s
mni_cortex 95 sample / s


In [16]:
%%bash

for space in schaefer400 flat mni_cortex; do
    for run in {1..8}; do
        path="results/epoch_times_${space}_${run}.txt"
        rm "${path}" 2>/dev/null
        logpath=$(echo output/input_space_v3/${space}_lr*_${run}/pretrain/log.txt)
        grep 'Train.*Total time' "${logpath}" | awk '{print $5}' > "$path"
    done
done


In [17]:
records = []
for space in input_spaces:
    for run in range(1, 9):
        path = f"results/epoch_times_{space}_{run}.txt"
        total_time = timedelta()
        with open(path) as f:
            for line in f:
                t = datetime.strptime(line.strip(), "%H:%M:%S")
                total_time += timedelta(hours=t.hour, minutes=t.minute, seconds=t.second)
        total_time = total_time.total_seconds()
        records.append({"space": space, "run": run, "time": total_time})

train_times = pd.DataFrame.from_records(records)
train_times = train_times.groupby("space").agg({"time": "median"})["time"].to_dict()
print(train_times)

{'flat': 100804.5, 'mni_cortex': 180771.0, 'schaefer400': 39973.0}


In [18]:
df = pd.DataFrame(
    {
        "train_time": train_times,
        "num_params": num_params,
        "flops": flops,
        "compute_tput": throughput,
        "load_tput": load_throughput,
    }
)
df["train_time"] = df["train_time"] / 3600
df["num_params"] = df["num_params"] / 1e6
df["flops"] = df["flops"] / 1e9
df = df.reset_index(drop=False, names="model")
print(df.to_string(index=False))
df.to_csv("results/input_space_compute.csv", index=False)

      model  train_time  num_params      flops  compute_tput   load_tput
       flat   28.001250   86.223360  92.111417    584.838876  274.423058
 mni_cortex   50.214167   86.990592 115.645788    472.415576  147.641602
schaefer400   11.103611   85.370880  89.001841    615.089925 3753.524353


In [19]:
space_names = {
    "schaefer400": "parcel",
    "flat": "flat",
    "mni_cortex": "volume",
}

In [20]:
df = pd.read_csv("results/input_space_compute.csv")

tput_scale = 16 / 1000  # k im / s

records = []
for _, row in df.iterrows():
    record = {
        "space": space_names[row["model"]],
        "time": f"{row['train_time']:.0f} hr",
        "params": f"{row['num_params']:.0f}M",
        "FLOPs": f"{row['flops']:.0f}G",
        "compute": f"{row['compute_tput'] * tput_scale:.0f}K fps",
        "data": f"{row['load_tput'] * tput_scale:.0f}K fps",
    }
    records.append(record)

df_fmt = pd.DataFrame.from_records(records)
df_fmt = df_fmt.iloc[[2, 0, 1]]
print(df_fmt.to_markdown(index=False))
print(df_fmt.to_latex(index=False))

| space   | time   | params   | FLOPs   | compute   | data    |
|:--------|:-------|:---------|:--------|:----------|:--------|
| parcel  | 11 hr  | 85M      | 89G     | 10K fps   | 60K fps |
| flat    | 28 hr  | 86M      | 92G     | 9K fps    | 4K fps  |
| volume  | 50 hr  | 87M      | 116G    | 8K fps    | 2K fps  |
\begin{tabular}{llllll}
\toprule
space & time & params & FLOPs & compute & data \\
\midrule
parcel & 11 hr & 85M & 89G & 10K fps & 60K fps \\
flat & 28 hr & 86M & 92G & 9K fps & 4K fps \\
volume & 50 hr & 87M & 116G & 8K fps & 2K fps \\
\bottomrule
\end{tabular}

